# 📅 Notebook 3: Sliding-Window Counter

The simplest limiter is a **fixed-window counter**: "max 5 requests per
calendar second". But that has a nasty flaw — a client can send 5 at
`0:59.99` and 5 more at `1:00.01`, for **10 req in 20ms**. Same window,
different clock, but real traffic doesn't respect wall clocks.

**Sliding-window log** fixes this by tracking *actual timestamps* of recent
requests. Used in production by Cloudflare, GitHub, Stripe.


## 🛠️ Setup

```bash
cd 04-patterns/rate-limiting-and-throttling
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## ❌ Bad: fixed-window counter — the boundary burst

A fixed window resets its counter on a **clock boundary** (`int(now // window)`). That's what makes it cheap — one integer per key in Redis, with a TTL — and it's what makes it wrong.

An attacker who knows where the boundary is spends their whole budget just *before* it and their whole next budget just *after* it. Two legal windows, one illegal burst.

Every limiter below takes an optional `now` so we can drive them from a real clock **and** replay recorded traces deterministically.

In [ ]:
import time
from collections import deque

class FixedWindow:
    """Counter that resets on clock-aligned boundaries — the naive limiter."""
    def __init__(self, limit, window):
        self.limit, self.window = limit, window
        self.window_id, self.count = None, 0

    def allow(self, now=None):
        now = time.monotonic() if now is None else now
        wid = int(now // self.window)        # which window are we in?
        if wid != self.window_id:            # rolled over -> counter resets to 0
            self.window_id, self.count = wid, 0
        if self.count < self.limit:
            self.count += 1
            return True
        return False

Now the attack, on the real clock — no poking at the limiter's internals. We wait until we're 100 ms from a window boundary, spend the whole budget, cross the boundary, and spend it again.

In [ ]:
def boundary_attack(limiter, window=1.0, burst=5):
    """Spend the budget just before a window edge, then again just after it."""
    # 1. sleep until we are ~100ms from the next boundary
    while window - (time.monotonic() % window) > 0.10:
        time.sleep(0.005)
    t0 = time.monotonic()
    allowed = sum(limiter.allow() for _ in range(burst))
    # 2. cross the boundary
    while time.monotonic() % window > window / 2:
        time.sleep(0.005)
    allowed += sum(limiter.allow() for _ in range(burst))
    return allowed, time.monotonic() - t0

n, span = boundary_attack(FixedWindow(limit=5, window=1.0))
print(f'fixed window (limit=5/s): {n} requests allowed in {span:.3f}s')
print(f'-> {n / span:.0f} req/s sustained through a limiter configured for 5 req/s '
      f'({n / 5:.0f}x the intended rate)')

## ✅ Better: sliding-window log — exact everywhere

Instead of a counter that resets, keep the **timestamps** of the recent allowed requests and count how many fall inside the trailing `window` seconds. There is no boundary to stand on, because the window moves with the request.

In [ ]:
class SlidingWindow:
    """Exact limiter: remembers when each allowed request happened."""
    def __init__(self, limit, window):
        self.limit, self.window = limit, window
        self.events = deque()

    def allow(self, now=None):
        now = time.monotonic() if now is None else now
        while self.events and self.events[0] <= now - self.window:
            self.events.popleft()            # drop what fell out of the window
        if len(self.events) < self.limit:
            self.events.append(now)          # only ALLOWED requests are recorded
            return True
        return False

# Same attack, same clock, same numbers — different answer.
n, span = boundary_attack(SlidingWindow(limit=5, window=1.0))
print(f'sliding log  (limit=5/s): {n} requests allowed in {span:.3f}s  <- capped correctly')

# And it still allows a well-behaved client its full budget once the window clears.
sw = SlidingWindow(limit=5, window=1.0)
print('5 quick requests    :', [int(sw.allow()) for _ in range(5)])
print('6th, immediately    :', int(sw.allow()))
time.sleep(1.01)
print('after the window ages out:', [int(sw.allow()) for _ in range(5)])

## 📊 Visualize it

Wall-clock loops make plots flaky, so from here we **replay a recorded arrival trace** through each limiter using the injected `now`. Same code paths, deterministic picture.

The trace is the attack repeated every second: 5 requests just before each boundary, 5 just after.

In [ ]:
import matplotlib.pyplot as plt

LIMIT, WINDOW = 5, 1.0

# Recorded arrivals: a burst of 5 at t=k-0.02 and 5 at t=k+0.02, for k = 1..4
arrivals = []
for k in range(1, 5):
    arrivals += [k - 0.02 + i * 0.001 for i in range(5)]
    arrivals += [k + 0.02 + i * 0.001 for i in range(5)]

def replay(limiter, trace):
    """Feed a recorded trace through a limiter; return the times it allowed."""
    return [t for t in trace if limiter.allow(t)]

fw_allowed = replay(FixedWindow(LIMIT, WINDOW), arrivals)
sw_allowed = replay(SlidingWindow(LIMIT, WINDOW), arrivals)

def rolling(allowed, window=WINDOW, step=0.02, end=5.0):
    """How many requests were allowed in the trailing `window` at each instant."""
    xs = [i * step for i in range(int(end / step) + 1)]
    return xs, [sum(1 for a in allowed if x - window < a <= x) for x in xs]

xf, yf = rolling(fw_allowed)
xs, ys = rolling(sw_allowed)

plt.figure(figsize=(9, 3))
plt.plot(xf, yf, color='red',   label=f'fixed window (peak={max(yf)})')
plt.plot(xs, ys, color='green', label=f'sliding log  (peak={max(ys)})')
plt.axhline(LIMIT, color='gray', linestyle='--', label=f'intended limit ({LIMIT})')
plt.xlabel('time (s)'); plt.ylabel('allowed in the trailing 1s')
plt.title('Fixed window lets through 2x the limit at every boundary')
plt.legend(); plt.tight_layout(); plt.show()

print(f'fixed window allowed {len(fw_allowed)}/{len(arrivals)}, peak {max(yf)} in any 1s window')
print(f'sliding log  allowed {len(sw_allowed)}/{len(arrivals)}, peak {max(ys)} in any 1s window')

## ⚖️ Hybrid: sliding-window counter (the "Cloudflare trick")

The log is exact but costs `O(N)` timestamps per key — at Cloudflare scale that's the whole budget. The hybrid keeps **two integers** (this window's count and the previous window's) and estimates the trailing window by assuming the previous window's traffic was spread **evenly**:

```
estimate = prev_count * (fraction of the previous window still inside the trailing window)
         + curr_count
```

O(1) memory, two `INCR`s in Redis, no boundary cliff. But read that assumption again — "spread evenly" is a guess, and it is the source of the error we measure below.

In [ ]:
class SlidingWindowCounter:
    """O(1) approximation used at scale by Cloudflare, Kong, etc."""
    def __init__(self, limit, window):
        self.limit, self.window = limit, window
        self.window_id = None
        self.curr = 0
        self.prev = 0

    def allow(self, now=None):
        now = time.monotonic() if now is None else now
        wid = int(now // self.window)
        if self.window_id is None:
            self.window_id = wid
        elif wid != self.window_id:
            # roll forward; if we skipped a whole window, the "previous" one is empty
            self.prev = self.curr if wid == self.window_id + 1 else 0
            self.curr = 0
            self.window_id = wid
        elapsed = now - wid * self.window                 # how far into this window
        weight = (self.window - elapsed) / self.window    # overlap with the previous one
        estimate = self.prev * weight + self.curr
        if estimate < self.limit:
            self.curr += 1
            return True
        return False

# Same boundary attack the fixed window failed:
n, span = boundary_attack(SlidingWindowCounter(limit=5, window=1.0))
print(f'sliding counter (limit=5/s): {n} allowed in {span:.3f}s')
print('-> no 2x cliff. Not exactly 5 either: the estimate is an approximation,')
print('   and the next cell measures exactly how wrong it can be.')

### 🔬 The price of the approximation

The estimate is only right when the previous window's traffic really was uniform. Replay two traces that are *not*, and measure how far off it lands:

- **front-loaded** — the previous window's traffic all happened at its very start, so it has *already* scrolled out of the trailing window. The counter still charges you for most of it → it **rejects requests it should allow**.
- **back-loaded** — the previous window's traffic all happened at its very end, so nearly all of it is still inside the trailing window. The counter discounts it → it **allows more than the limit**.

In [ ]:
LIMIT, WINDOW = 20, 1.0

def peak_in_any_window(times, window=WINDOW):
    return max((sum(1 for x in times if t - window < x <= t) for t in times), default=0)

traces = {
    # previous window's 20 requests at its START, then 20 more just after the boundary
    'front-loaded': [0.00 + i * 0.001 for i in range(LIMIT)] +
                    [1.02 + i * 0.001 for i in range(LIMIT)],
    # previous window's 20 requests at its END, then 20 more just after the boundary
    'back-loaded ': [0.97 + i * 0.001 for i in range(LIMIT)] +
                    [1.02 + i * 0.001 for i in range(LIMIT)],
}

print(f'{"trace":<14}{"exact log":>22}{"O(1) counter":>22}')
for name, trace in traces.items():
    ex = replay(SlidingWindow(LIMIT, WINDOW), trace)
    ap = replay(SlidingWindowCounter(LIMIT, WINDOW), trace)
    print(f'{name:<14}'
          f'{f"{len(ex)} allowed, peak {peak_in_any_window(ex)}":>22}'
          f'{f"{len(ap)} allowed, peak {peak_in_any_window(ap)}":>22}')

print(f'\nlimit = {LIMIT}/s')
print('front-loaded: the counter allowed 21 where the exact limiter allowed 40 —')
print('              it rejected 19 requests that were entirely within the limit.')
print('back-loaded : the counter peaked at 21 in one second — 5% OVER the limit.')
print('\nCloudflare measured ~0.003% of requests wrongly allowed on real traffic.')
print('That is a fine trade for O(1) memory on a login endpoint. It is not fine')
print('for a hard contractual quota you bill against — use the log (or GCRA) there.')

## 📊 Algorithm comparison

| Algorithm | Memory / key | Burst behavior | Boundary safe? | Exact? | Typical use |
|---|---|---|---|---|---|
| Fixed window counter | 1 int | up to **2× limit** at the boundary | ❌ | ❌ | never, once you know better |
| Sliding window log | N timestamps | smooth, exact | ✅ | ✅ | billable quotas, low-cardinality keys |
| Sliding window counter | 2 ints | ~exact (few % error either way) | ✅ | ❌ | edge/CDN scale, huge key counts |
| Token bucket | 2 floats | bounded burst by design | ✅ | ✅ | user-facing API limits |
| Leaky bucket / GCRA | 1–2 floats | none (paced) | ✅ | ✅ | shaping to a fragile downstream |

### 🧠 When to use which

- **Sliding window log** — when being wrong costs money or trust: billed quotas, auth/anti-abuse counters, anything a customer can audit. Cost: memory grows with the limit, and eviction needs care (a Redis sorted set per key, trimmed on every call).
- **Sliding window counter** — when you have millions of keys and a few percent of error is cheaper than the memory. Cost: it can both over- and under-admit, as measured above, and you can't explain a specific decision to a customer.
- **Fixed window** — only when the boundary burst genuinely doesn't matter (coarse "1M/month" quotas). Its one real virtue is that it's a single `INCR` + `EXPIRE`.
- **Not a rate limiter at all** — if what you're protecting is a finite resource (DB connections, memory) rather than a rate, you want a **concurrency limit**. Notebook 5.

For a **distributed** limiter (many servers, one shared counter), replace in-memory state with **Redis** (`INCR` + TTL for fixed window, sorted sets for a log, two counters for the hybrid). That's notebook 4 — including the race you get if you do it naively.